# 🫀 Cardiovascular Disease Risk Prediction
**Author:** Manav Verma  
**Dataset:** Healthcare Synthetic Data (15,000 records, 19 features)  
**Objective:** Predict cardiovascular/heart disease risk using EDA + Machine Learning.

---

## Table of Contents
1. Import Libraries
2. Load & Explore Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Model Building
6. Model Evaluation & Comparison
7. Feature Importance
8. Conclusion


---
## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='deep')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)
print('All libraries imported successfully!')


---
## 2. Load & Explore Dataset


In [ ]:
df = pd.read_csv('dataset/healthcare_synthetic_data.csv')
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()


In [ ]:
print('=== Dataset Info ===')
df.info()


In [ ]:
df.describe().round(2)


In [ ]:
print('Missing Values:', df.isnull().sum().sum())
print('Duplicate Rows:', df.duplicated().sum())


In [ ]:
target_counts = df['Heart_Disease_Risk'].value_counts()
labels = ['Low Risk (0)', 'High Risk (1)']
colors = ['#2ecc71', '#e74c3c']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bars = axes[0].bar(labels, target_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, target_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count}\n({count/len(df)*100:.1f}%)', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')
axes[0].set_title('Heart Disease Risk Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, max(target_counts.values) * 1.2)
axes[1].pie(target_counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proportion of Risk Classes', fontsize=14, fontweight='bold')
plt.suptitle('Target Variable: Heart Disease Risk', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 3. Exploratory Data Analysis (EDA)


In [ ]:
numerical_cols = ['Age', 'Height_cm', 'Weight_kg', 'BMI', 'Systolic_BP', 'Diastolic_BP',
                  'Cholesterol_Total', 'Cholesterol_LDL', 'Cholesterol_HDL',
                  'Fasting_Blood_Sugar', 'Stress_Level', 'Sleep_Hours']
categorical_cols = ['Gender', 'Smoking_Status', 'Alcohol_Consumption',
                    'Physical_Activity_Level', 'Family_History']
print(f'Numerical ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical ({len(categorical_cols)}): {categorical_cols}')


In [ ]:
# 3.1 Distribution of Numerical Features
fig, axes = plt.subplots(4, 3, figsize=(18, 18))
axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    for rv, color, lbl in zip([0, 1], ['#2ecc71', '#e74c3c'], ['Low Risk', 'High Risk']):
        axes[i].hist(df[df['Heart_Disease_Risk'] == rv][col],
                     bins=30, alpha=0.6, color=color, label=lbl, edgecolor='white')
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col); axes[i].set_ylabel('Frequency'); axes[i].legend(fontsize=9)
plt.suptitle('Distribution of Numerical Features by Heart Disease Risk',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# 3.2 Boxplots — Numerical Features vs Risk
fig, axes = plt.subplots(4, 3, figsize=(18, 18))
axes = axes.flatten()
palette = {0: '#2ecc71', 1: '#e74c3c'}
for i, col in enumerate(numerical_cols):
    sns.boxplot(data=df, x='Heart_Disease_Risk', y=col, palette=palette,
                ax=axes[i], linewidth=1.5)
    axes[i].set_title(f'{col} vs Risk', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Heart Disease Risk (0=Low, 1=High)')
    axes[i].set_ylabel(col)
plt.suptitle('Boxplots: Numerical Features vs Heart Disease Risk',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# 3.3 Categorical Features vs Risk
cat_labels = {
    'Gender': {0: 'Female', 1: 'Male'},
    'Smoking_Status': {0: 'Non-Smoker', 1: 'Smoker'},
    'Alcohol_Consumption': {0: 'None', 1: 'Moderate', 2: 'Heavy'},
    'Physical_Activity_Level': {0: 'Sedentary', 1: 'Low', 2: 'Moderate', 3: 'High'},
    'Family_History': {0: 'No History', 1: 'Has History'}
}
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(categorical_cols):
    cross = pd.crosstab(df[col], df['Heart_Disease_Risk'], normalize='index') * 100
    cross.index = [cat_labels[col].get(k, str(k)) for k in cross.index]
    cross.columns = ['Low Risk', 'High Risk']
    cross.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'],
               edgecolor='white', rot=30)
    axes[i].set_title(f'{col} vs Heart Disease Risk', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
axes[-1].axis('off')
plt.suptitle('Categorical Features vs Heart Disease Risk',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# 3.4 Correlation Heatmap
plt.figure(figsize=(16, 12))
corr_df = df.drop(columns=['Patient_ID'])
corr = corr_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, center=0, vmin=-1, vmax=1,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


In [ ]:
# 3.5 Feature Correlation with Target
corr_target = corr_df.corr()['Heart_Disease_Risk'].drop('Heart_Disease_Risk').sort_values()
colors_bar = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_target]
plt.figure(figsize=(10, 7))
plt.barh(corr_target.index, corr_target.values, color=colors_bar, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('Correlation with Heart Disease Risk', fontsize=12)
plt.title('Feature Correlations with Target Variable', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# 3.6 Age vs BMI Scatter
plt.figure(figsize=(10, 7))
scatter = plt.scatter(df['Age'], df['BMI'],
                      c=df['Heart_Disease_Risk'], cmap='RdYlGn_r',
                      alpha=0.4, s=20, edgecolors='none')
plt.colorbar(scatter, label='Heart Disease Risk (0=Low, 1=High)')
plt.xlabel('Age', fontsize=12); plt.ylabel('BMI', fontsize=12)
plt.title('Age vs BMI — Colored by Heart Disease Risk', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# 3.7 Blood Pressure Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for rv, color, lbl in zip([0, 1], ['#2ecc71', '#e74c3c'], ['Low Risk', 'High Risk']):
    subset = df[df['Heart_Disease_Risk'] == rv]
    axes[0].hist(subset['Systolic_BP'], bins=30, alpha=0.6, color=color, label=lbl, edgecolor='white')
    axes[1].hist(subset['Diastolic_BP'], bins=30, alpha=0.6, color=color, label=lbl, edgecolor='white')
axes[0].set_title('Systolic BP by Risk', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Systolic BP (mmHg)'); axes[0].legend()
axes[1].set_title('Diastolic BP by Risk', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Diastolic BP (mmHg)'); axes[1].legend()
plt.suptitle('Blood Pressure Analysis by Heart Disease Risk', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 4. Data Preprocessing


In [ ]:
df_model = df.drop(columns=['Patient_ID'])

# Feature Engineering
df_model['Pulse_Pressure']   = df_model['Systolic_BP'] - df_model['Diastolic_BP']
df_model['Cholesterol_Ratio'] = df_model['Cholesterol_LDL'] / df_model['Cholesterol_HDL']
df_model['BMI_Category'] = pd.cut(
    df_model['BMI'], bins=[0, 18.5, 24.9, 29.9, 100], labels=[0, 1, 2, 3]
).astype(int)

print(f'Shape after feature engineering: {df_model.shape}')
print('New features: Pulse_Pressure, Cholesterol_Ratio, BMI_Category')


In [ ]:
X = df_model.drop(columns=['Heart_Disease_Risk'])
y = df_model['Heart_Disease_Risk']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('StandardScaler applied.')

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
print(f'After SMOTE : {dict(pd.Series(y_train_bal).value_counts())}')


---
## 5. Model Building


In [ ]:
models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors':    KNeighborsClassifier(n_neighbors=7),
    'Decision Tree':          DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':                XGBClassifier(n_estimators=100, random_state=42,
                                            eval_metric='logloss', verbosity=0),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42)
}

results = {}
print('Training models...\n')
for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob)
    }
    r = results[name]
    print(f"{name:<28} | Acc: {r['Accuracy']:.4f} | F1: {r['F1-Score']:.4f} | AUC: {r['ROC-AUC']:.4f}")
print('\nAll models trained!')


---
## 6. Model Evaluation & Comparison


In [ ]:
metrics_df = pd.DataFrame({
    name: {k: v for k, v in info.items()
           if k in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']}
    for name, info in results.items()
}).T.round(4).sort_values('ROC-AUC', ascending=False)

display(metrics_df.style
        .background_gradient(cmap='RdYlGn',
                             subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])
        .format('{:.4f}'))


In [ ]:
# Bar chart comparison
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names  = list(metrics_df.index)
x = np.arange(len(model_names)); width = 0.15
fig, ax = plt.subplots(figsize=(16, 7))
palette = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
for i, (metric, color) in enumerate(zip(metric_names, palette)):
    vals = [metrics_df.loc[m, metric] for m in model_names]
    ax.bar(x + i*width, vals, width, label=metric, color=color, alpha=0.85, edgecolor='white')
ax.set_xlabel('Model', fontsize=12); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x + width*2); ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=10)
ax.set_ylim(0, 1.1); ax.legend(fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# ROC Curves
plt.figure(figsize=(10, 8))
roc_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']
for (name, info), color in zip(results.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_test, info['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={info['ROC-AUC']:.4f})", color=color, linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=9); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
best_model_name = metrics_df.index[0]
best_info = results[best_model_name]
print(f'Best Model: {best_model_name}')

cm = confusion_matrix(y_test, best_info['y_pred'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'],
            linewidths=1.5, linecolor='white', ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label'); axes[0].set_ylabel('True Label')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'],
            linewidths=1.5, linecolor='white', ax=axes[1])
axes[1].set_title(f'Normalized CM — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(classification_report(y_test, best_info['y_pred'],
                             target_names=['Low Risk', 'High Risk']))


In [ ]:
# Cross-Validation
print(f'5-Fold Cross-Validation — {best_model_name}')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_info['model'], X_train_bal, y_train_bal,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'CV AUC Scores: {[round(s, 4) for s in cv_scores]}')
print(f'Mean AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')


---
## 7. Feature Importance


In [ ]:
tree_models  = ['Random Forest', 'Gradient Boosting', 'XGBoost']
feature_names = list(X.columns)
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for i, model_name in enumerate(tree_models):
    model = results[model_name]['model']
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:12]
    colors_fi = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 12))
    axes[i].barh([feature_names[j] for j in reversed(top_idx)],
                 importances[list(reversed(top_idx))],
                 color=colors_fi[::-1], edgecolor='white')
    axes[i].set_title(f'{model_name}\nTop 12 Features', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Importance Score')
plt.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# Aggregated importance
agg_imp = np.zeros(len(feature_names))
for mn in tree_models:
    agg_imp += results[mn]['model'].feature_importances_
agg_imp /= len(tree_models)

fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': agg_imp})
fi_df = fi_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
bar_colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(fi_df)))
plt.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1],
         color=bar_colors[::-1], edgecolor='white')
plt.xlabel('Average Importance Score', fontsize=12)
plt.title('Aggregated Feature Importance (RF + GBM + XGBoost)',
          fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('Top 5 Most Important Features:')
for _, row in fi_df.head(5).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")


---
## 8. Conclusion


In [ ]:
print('=' * 65)
print('  CARDIOVASCULAR DISEASE RISK PREDICTION - PROJECT SUMMARY')
print('=' * 65)
print('\nDataset : 15,000 patients x 19 features (no missing values)')
print('Target  : Heart_Disease_Risk (0 = Low Risk, 1 = High Risk)')
print(f'\nBest Model: {best_model_name}')
print(f"  Accuracy : {best_info['Accuracy']:.2%}")
print(f"  Precision: {best_info['Precision']:.2%}")
print(f"  Recall   : {best_info['Recall']:.2%}")
print(f"  F1-Score : {best_info['F1-Score']:.2%}")
print(f"  ROC-AUC  : {best_info['ROC-AUC']:.4f}")
print('\nKey Findings:')
print('  - Age, Systolic BP, BMI and Cholesterol are top risk factors')
print('  - Smokers and those with family history show significantly higher risk')
print('  - Physical activity and adequate sleep reduce cardiovascular risk')
print('  - SMOTE improved model sensitivity for the high-risk class')
print('  - Ensemble models (XGBoost, RF, GBM) outperformed simpler models')
print('\nProject Complete! — Manav Verma')
print('=' * 65)
